# Week 4: Heston Stochastic Volatility

## Why we're here

Dupire (Week 3) fits a smile *snapshot* by letting $\sigma_{loc}(S,t)$ vary deterministically
with spot and time. It reproduces today's market prices exactly, but it gets the **dynamics**
wrong: as spot moves, Dupire's smile rolls in a direction inconsistent with what's actually
observed in markets (this was flagged from the real SPX data in Week 1, where 30 DTE skew was
observed ~70% steeper than 75 DTE, i.e. a genuine term structure, not a fixed shape). The reason
is structural: in Dupire's world $S_t$ alone is Markovian, $\sigma_{loc}$ is just a function
evaluated along whatever path $S$ takes, so there's no independent source of randomness driving
volatility itself.

Heston fixes this by giving variance its own stochastic process, correlated with spot but not
determined by it:

$$dS_t = rS_t\,dt + \sqrt{v_t}\,S_t\,dW_t^S$$
$$dv_t = \kappa(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW_t^v, \qquad dW_t^S\,dW_t^v = \rho\,dt$$

Five parameters: $v_0$ (initial variance), $\kappa$ (mean-reversion speed), $\theta$ (long-run
variance), $\xi$ (vol-of-vol), $\rho$ (spot-vol correlation, typically negative for equities,
this is what generates skew *endogenously* rather than by construction as in Dupire).

Consequence: $(S_t, v_t)$ jointly is Markovian, but $S_t$ alone is not. Feynman-Kac now gives a
PDE in three variables $(t, S, v)$ rather than Dupire's two, which is part of why this week
leans on characteristic-function pricing (Week 5) rather than a direct PDE solve.

**The known limitation to watch for going forward**: Heston is still Markovian in $(S,v)$, so
even though it fixes the *smile-dynamics* problem, it flattens the term structure of skew too
quickly at long maturities, a real but different failure mode from Dupire's. This is what
eventually motivates jumps (Week 7) and rough volatility (Week 9).

## The CIR thread (payoff)

The variance process on its own is the Cox-Ingersoll-Ross (CIR) process, flagged back on Day 7
and Day 10. Its diffusion coefficient $\xi\sqrt{v_t}$ is state-dependent: as $v_t \to 0$ the
noise shrinks and the mean-reverting drift $\kappa\theta\,dt$ dominates, which is what keeps
$v_t$ nonnegative (contrast with an OU process, constant diffusion coefficient, Gaussian at
every fixed time, can go negative, fine for rates, wrong for variance).

This state-dependence is exactly why CIR's transition density is a scaled **noncentral
chi-squared** distribution rather than Gaussian, closed form, and exact (no discretization
bias) simulation is possible directly from it.

**Feller condition**: $2\kappa\theta \geq \xi^2$ guarantees the exact continuous-time process
never touches zero. It says nothing about discrete-time approximations of it, see lessons below.

## Plan for this week

1. Euler-discretized simulator with truncation, get something running end to end
2. Diagnose and fix the negativity failure mode (see lessons learned)
3. Andersen QE scheme (better bias/speed tradeoff than truncated Euler)
4. Characteristic function derivation, sets up Week 5's Fourier pricing directly


## Lessons learned (Day 16)

- **$W_t^S$, $W_t^v$ notation**: the superscript labels which SDE a Brownian motion belongs to,
  not an exponent. Correlation $dW_t^S\,dW_t^v = \rho\,dt$ is the continuous-time quadratic
  covariation, constructible from two independent Brownians via
  $W^S = W^1$, $W^v = \rho W^1 + \sqrt{1-\rho^2}\,W^2$. This is also the Cholesky recipe for
  simulation.

- **OU vs CIR**: OU has constant diffusion coefficient $\xi$, Gaussian at every fixed time, can
  go negative. CIR's diffusion coefficient $\xi\sqrt{v_t}$ vanishes as $v_t \to 0$, which is what
  keeps the process nonnegative and is also what breaks Gaussianity, producing the noncentral
  chi-squared transition density instead.

- **Euler discretization of CIR fails structurally, not just occasionally**: the update
  $v_{t+\Delta t} = v_t + \kappa(\theta - v_t)\Delta t + \xi\sqrt{v_t}\sqrt{\Delta t}\,Z$ uses an
  unbounded Gaussian shock evaluated at the *start* of the interval. For any $\Delta t > 0$ there
  is nonzero probability $Z$ drives the update below zero, at which point the next step needs
  $\sqrt{\text{negative}}$. Feller only makes this rarer (less time spent near zero on average),
  it does not make it impossible for finite $\Delta t$, the failure mode is present regardless of
  parameters.

- **Exact simulation cost, two separate issues**:
  1. Sampling $v_{t+\Delta t}$ exactly requires a noncentral chi-squared draw: Poisson-mixture of
     central chi-squares, $N \sim \text{Poisson}(\lambda/2)$, then central $\chi^2$ with
     $d + 2N$ degrees of freedom. Real overhead per step vs. Euler's single Gaussian pull.
  2. Exact $(S_t, v_t)$ joint simulation needs more than the two variance endpoints. The spot SDE's
     noise term integrates $\sqrt{v_s}$ over the *whole step*, so exact joint simulation
     (Broadie-Kaya) requires sampling the integrated variance
     $\int_t^{t+\Delta t} v_s\,ds$ conditional on both endpoints, which has no closed-form
     distribution and is inverted numerically from its Laplace transform. This is why full
     Broadie-Kaya is rarely used in practice.

- **Decision**: Andersen QE next (moment-matched scheme approximating the true conditional
  distribution with a cheap-to-sample one), better bias/speed tradeoff than truncated Euler,
  much cheaper than full Broadie-Kaya. Implementation Day 17.
